# 性能评估与加速比损耗分析



In [87]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
import re


def _full_lifecycle_segments(csv):
    """
    - start_ms: 甘特图起始位
    - instance_jitter: 抖动分析
    - total_cycle: 加速比分析
    - 各阶段 lat: 风琴图/小提琴图分析
    """
    if not os.path.exists(csv): return pd.DataFrame()
    df = pd.read_csv(csv).sort_values(["tid", "seq"])
    t_min = df["t_us"].min() # 全局起始时间锚点
    excl = {"timer", "main::expand", "main::rollover"}
    rows = []

    for tid, g in df.groupby("tid"):
        events = g.copy()
        for i in range(len(events)):
            curr = events.iloc[i]
            if curr["kind"] == "execute" and curr["tag"] not in excl:
                try:
                    prev = events.iloc[:i]
                    rel = prev[prev["kind"] == "release"].iloc[-1]
                    wak = prev[prev["kind"] == "wake"].iloc[-1]
                    nxt = events.iloc[i+1:]
                    com = nxt[nxt["kind"] == "complete"].iloc[0]
                    fin = nxt[nxt["kind"] == "finished"].iloc[0]
                    slp = nxt[nxt["kind"] == "sleep"].iloc[0]

                    # 计算各阶段耗时
                    w_lat = rel["t_us"] - wak["t_us"]
                    p_lat = curr["t_us"] - rel["t_us"]
                    a_lat = com["t_us"] - curr["t_us"]
                    o_lat = fin["t_us"] - com["t_us"]
                    s_lat = slp["t_us"] - fin["t_us"]

                    rows.append({
                        "tid": str(tid),
                        "algo": curr["tag"],
                        "seq": curr["seq"],
                        # 1. 甘特图所需：相对于全局起点的毫秒数
                        "start_ms": (wak["t_us"] - t_min) / 1000.0,
                        # 2. 算法执行时长(ms)
                        "algo_ms": a_lat / 1000.0,
                        # 3. 各阶段原始数据(us)
                        "wakeup_lat": w_lat,
                        "pack": p_lat,
                        "algo_us": a_lat,
                        "post": o_lat,
                        "sleep_lat": s_lat,
                        # 4. 抖动与加速比指标
                        "instance_jitter": w_lat + p_lat + o_lat + s_lat,
                        "total_cycle": slp["t_us"] - wak["t_us"]
                    })
                except: continue
    return pd.DataFrame(rows)


# 加载数据
CSV_PATH = "temp/tracing.csv"
full_job_df = _full_lifecycle_segments(CSV_PATH)

In [88]:
# --- Science 论文专用：沉稳低饱和度配色 ---
SCI_PALETTE = {
    "Active": "#3B5998",    # 灰蓝 (Deep Blue)
    "Overhead": "#A6192E",  # 砖红 (Muted Red)
    "Idle": "#8E8E8E",      # 中灰 (Middle Gray)
    "Wakeup": "#5E8B61",    # 鼠尾草绿 (Sage Green)
    "Pack": "#709BFF",      # 浅钢蓝 (Light Blue)
    "Post": "#D6A531",      # 暗金 (Old Gold)
    "Sleep": "#4D4D4D"      # 深碳色 (Charcoal)
}

# 风格：经典、高对比度、各指标定义明确
PALETTE_NPG = {
    "Active": "#00468B",    # 经典的深海蓝 (Deep Navy)
    "Overhead": "#AD002A",  # 警告感的暗红 (Brick Red)
    "Idle": "#ADB6B6",      # 带有质感的浅灰 (Silver Gray)
    "Wakeup": "#42B540",    # 专业的草绿 (Meadow Green)
    "Pack": "#0099B4",      # 清爽的青色 (Cyan Blue)
    "Post": "#925E9F",      # 典雅的紫色 (Amethyst)
    "Sleep": "#1B1919"      # 近乎黑的深灰 (Deep Charcoal)
}

# 风格：极简、现代、强调“干活”与“空闲”的二元对立
PALETTE_SCIENCE = {
    "Active": "#3B5998",    # 灰调蓝 (Low-sat Blue)
    "Overhead": "#D7191C",  # 哑光红 (Muted Crimson)
    "Idle": "#D9D9D9",      # 极浅灰（背景感强）
    "Wakeup": "#7570B3",    # 蓝紫色 (Dusty Purple)
    "Pack": "#66A61E",      # 橄榄绿 (Olive Green)
    "Post": "#E6AB02",      # 芥末黄 (Mustard Gold)
    "Sleep": "#666666"      # 标准中灰
}

# 风格：冷峻、工业感、适合体现“底层架构”
PALETTE_HPC = {
    "Active": "#1F77B4",    # 钢蓝色 (Steel Blue)
    "Overhead": "#B2182B",  # 铁锈红 (Rust)
    "Idle": "#E0E0E0",      # 浅铝灰 (Aluminium)
    "Wakeup": "#2166AC",    # 另一种深蓝，与 Active 呼应
    "Pack": "#67A9CF",      # 浅天蓝 (Sky Blue)
    "Post": "#F4A582",      # 浅砖色 (Light Terracotta)
    "Sleep": "#404040"      # 沥青黑 (Asphalt)
}

# 风格：国际标准、最高辨识度、色盲友好
PALETTE_SAFE = {
    "Active": "#0072B2",    # 极致蓝 (Safe Blue)
    "Overhead": "#D55E00",  # 琥珀橙 (Amber)
    "Idle": "#F0E442",      # 浅米黄 (Pale Yellow)
    "Wakeup": "#009E73",    # 蓝绿色 (Bluish Green)
    "Pack": "#56B4E9",      # 浅蓝色 (Light Sky)
    "Post": "#CC79A7",      # 莫兰迪粉 (Dusty Pink)
    "Sleep": "#000000"      # 纯黑
}

SCI_PALETTE = SCI_PALETTE

## 1. 任务甘特图 (Gantt Chart) —— 核心忙闲视图

说明：不同于散点图，甘特图用“条块”展示任务。你能清晰看到任务的实际跨度以及核心之间的并行间隙。

In [89]:
def draw_px_gantt_sci(df_jobs):
    fig = px.bar(
        df_jobs,
        base="start_ms",
        x=df_jobs["algo_us"]/1000.0, # 转换为ms
        y="tid",
        color="algo",
        orientation='h',
        color_discrete_sequence=px.colors.qualitative.Prism, # 自动选择不艳丽的序列
        title="Hardware Thread Gantt Chart (Execution Continuity Analysis)",
        labels={"base": "Time (ms)", "x": "Duration (ms)", "tid": "Thread ID"}
    )

    fig.update_layout(
        plot_bgcolor='rgba(240,240,240,0.5)', # 淡淡的灰底
        xaxis=dict(showgrid=True, gridcolor='white'),
        yaxis=dict(showgrid=True, gridcolor='white'),
        font=dict(family="Arial", size=12)
    )
    fig.show()

draw_px_gantt_sci(full_job_df)

## 2. 负载构成分析 (Science 风格饼图)
说明：使用 Science 配色，分析次数比例与时间贡献。

In [90]:
def draw_px_composition_sci(df_jobs):
    stats = df_jobs.groupby("algo").agg(count=("algo", "count"), total_time=("algo_us", "sum")).reset_index()

    # 使用灰调配色序列
    muted_colors = ["#334E6F", "#58728A", "#8195A8", "#ABB9C8", "#D5DDE4"]

    fig = px.pie(stats, values='total_time', names='algo',
                 title="Effective Workload Distribution (Time-based)",
                 color_discrete_sequence=muted_colors)
    fig.update_traces(textinfo='percent+label', hole=0.3)
    fig.show()

draw_px_composition_sci(full_job_df)

## 3. 核心效率拆解图 (Science Stacked Bar)
说明：将时间切分为：有效功、开销、空闲。这是论文中证明“独占效率”的关键图表。

In [99]:
def draw_px_efficiency_sci(df_jobs):
    total_obs = df_jobs["start_ms"].max() * 1000 + df_jobs["instance_jitter"].max()
    res = []
    for tid in df_jobs['tid'].unique():
        sub = df_jobs[df_jobs['tid'] == tid]
        active = sub['algo_us'].sum()
        ovhd = sub['instance_jitter'].sum()
        idle = max(0, total_obs - (active + ovhd))

        res.append({"TID": tid, "State": "1. Active", "Time": active})
        res.append({"TID": tid, "State": "2. Sched Overhead", "Time": ovhd})
        res.append({"TID": tid, "State": "3. Wait/Idle", "Time": idle})

    fig = px.bar(
        pd.DataFrame(res), y="TID", x="Time", color="State",
        color_discrete_map={
            "1. Active": SCI_PALETTE["Active"],
            "2. Sched Overhead": SCI_PALETTE["Overhead"],
            "3. Wait/Idle": SCI_PALETTE["Idle"]
        },
        orientation='h',
        title="Core Resource Allocation Efficiency"
    )

    fig.update_layout(
        plot_bgcolor='white',
        xaxis_title="Accumulated Time (us)",
        barmode='stack',
        font=dict(size=12)
    )
    fig.show()

draw_px_efficiency_sci(full_job_df)

## 4. 调度抖动小提琴图 (Interactive Jitter)
说明：展示 extra_overhead 的分布，点位展示了异常值的分布

In [92]:
def draw_px_jitter_violin(df_jobs):
    # 限制分析，只看算法相关的调度抖动
    fig = px.violin(
        df_jobs,
        x="algo",
        y="instance_jitter",
        color="algo",
        box=True,
        points="outliers", # 仅显示离群点，防止点太多卡顿
        color_discrete_sequence=[SCI_PALETTE["Active"]],
        title="Per-instance Scheduling Jitter Distribution",
        labels={"instance_jitter": "Instance Overhead (us)", "algo": "Algorithm"}
    )

    fig.update_layout(
        plot_bgcolor='white',
        yaxis=dict(
            gridcolor='#F0F0F0',
            title="Jitter (us)",
            zerolinecolor='#D0D0D0'
        ),
        xaxis=dict(
            title="Algorithm Nodes",
            linecolor='#D0D0D0',
            tickangle=45         # 如果标签太密，旋转45度
        ),
        showlegend=False,
        height=500
    )

    mean_val = df_jobs["instance_jitter"].mean()
    fig.add_hline(y=mean_val,
                  line_dash="dot",
                  line_color=SCI_PALETTE["Idle"],
                  annotation_text=f"Mean: {mean_val:.1f}us")

    fig.show()

draw_px_jitter_violin(full_job_df)

In [93]:
def draw_px_jitter_violin_p99(df_jobs):
    """
    绘制过滤掉 P99 极端值后的抖动分布图
    使用 transform 方法确保不会丢失 'algo' 列
    """
    if df_jobs.empty:
        print("Error: DataFrame is empty.")
        return

    # 1. 核心修复：使用 transform 计算每个 algo 分组的 P99 阈值
    # 这样可以生成一个与原 df 行数一致的布尔遮罩，不会破坏列结构
    p99_thresholds = df_jobs.groupby("algo")["instance_jitter"].transform(lambda x: x.quantile(0.99))
    df_filtered = df_jobs[df_jobs["instance_jitter"] <= p99_thresholds].copy()

    # 2. 打印列名以供调试（如果还报错，可以看到当前的列状态）
    # print("Current columns:", df_filtered.columns.tolist())

    # 3. 绘图
    fig = px.violin(
        df_filtered,
        x="algo",                  # 现在的 'algo' 确定是一列
        y="instance_jitter",
        color="algo",
        box=True,
        points="outliers",
        color_discrete_sequence=[SCI_PALETTE["Active"]],
        title="Per-instance Jitter Distribution (Filtered > P99)",
        labels={"instance_jitter": "Jitter (us)", "algo": "Algorithm"}
    )

    # 4. 论文风格布局优化 (Science Style)
    fig.update_layout(
        plot_bgcolor='white',
        yaxis=dict(
            gridcolor='#F0F0F0',
            title="Jitter (us)",
            zerolinecolor='#D0D0D0'
        ),
        xaxis=dict(
            title="Algorithm Nodes",
            linecolor='#D0D0D0',
            tickangle=45         # 如果标签太密，旋转45度
        ),
        showlegend=False,
        height=500
    )

    # 添加全局均值参考线
    mean_val = df_filtered["instance_jitter"].mean()
    fig.add_hline(y=mean_val,
                  line_dash="dot",
                  line_color=SCI_PALETTE["Idle"],
                  annotation_text=f"Mean: {mean_val:.1f}us")

    fig.show()

# 执行
draw_px_jitter_violin_p99(full_job_df)

## 5. 生命周期风琴图 (Mean Lifecycle)
说明：Science 风格的各阶段平均占比。

In [94]:
def draw_px_accordion_sci(df_jobs):
    cols = ["wakeup_lat", "pack", "algo_us", "post", "sleep_lat"]
    g = df_jobs.groupby("algo")[cols].mean().reset_index()
    long_g = g.melt(id_vars="algo", value_vars=cols, var_name="Stage", value_name="Avg_Time_us")

    fig = px.bar(
        long_g, y="algo", x="Avg_Time_us", color="Stage",
        orientation='h',
        color_discrete_map={
            "wakeup_lat": SCI_PALETTE["Wakeup"],
            "pack": SCI_PALETTE["Pack"],
            "algo_us": SCI_PALETTE["Active"],
            "post": SCI_PALETTE["Post"],
            "sleep_lat": SCI_PALETTE["Sleep"]
        },
        title="Average Task Cycle Decomposition"
    )

    fig.update_layout(
        plot_bgcolor='white',
        barmode='stack',
        xaxis_title="Time (us)",
        legend_title="Lifecycle Stage"
    )
    fig.show()

draw_px_accordion_sci(full_job_df)

### 1. 任务分布 (Gantt Chart)
甘特图揭示了任务执行的**关键路径**。如果某核心（TID）的蓝色条块之间有明显的空白，说明调度器在此时刻没有待处理任务，或者受到了主线程 `rollover` 逻辑的阻塞。

### 2. 执行次数比例与贡献 ($P_i$ & $P_i \times C_i$)
在 Science 论文中，我们必须区分“调用频繁”的任务和“占用时间”的任务。
- **高频低耗 ($P_i$ 高)**: 容易引入累计调度开销。
- **低频高耗 ($P_i \times C_i$ 高)**: 它是系统并行度的上限（Amdahl瓶颈）。

### 3. 核心效率与气泡 (Efficiency & Bubbles)
核心独占（Core Pinning）的成本是放弃了系统的通用调度灵活性。
- **蓝色 (Active)** 比例越高，说明独占策略越成功。
- **灰色 (Idle)** 代表任务流不足，核心在 `cv_wait`。
- **红色 (Overhead)** 代表全局队列的锁竞争（Lock Contention）导致的非生产性耗时。

### 4. 调度抖动 (Scheduling Jitter)
- **小提琴图**展示了每个算法节点的调度抖动
- **P99 过滤**可以去除极端值，帮助我们更好地理解大多数任务的调度行为。

# 加速比与扩展性分析 (Speedup & Scalability)

### 1. 核心定义
* **Speedup $S(m)$**: $S(m) = \frac{T_{hp}(1)}{T_{hp}(m)}$
  - $T_{hp}(1)$: 单核心运行整个超周期（Makespan）的中位时间。
  - $T_{hp}(m)$: $m$ 个核心并行运行时，完成同样任务量的时间。
* **Parallel Efficiency $E(m)$**: $E(m) = \frac{S(m)}{m} \times 100\%$
  - 反映了每增加一个核心，带来的边际收益。理想状态应为 100%。

### 2. 科学预期
* **Linear Speedup**: $S(m) = m$（理想情况）。
* **Sub-linear Speedup**: 由于调度开销、锁竞争、释放受限（Release-limited），实际曲线会低于理想线并逐渐趋于饱和。

In [123]:
def extract_hp_makespans(csv):
    if not os.path.exists(csv): return pd.DataFrame()
    # 读取数据，确保 t_us 为数值
    df = pd.read_csv(csv)
    df["t_us"] = pd.to_numeric(df["t_us"], errors='coerce')
    df = df.dropna(subset=["t_us"]).sort_values("t_us")

    # 1. 获取超周期边界 (main::rollover 的 release 时刻)
    # 使用 str.contains 增加容错性，防止标签有微小差异
    is_rollover = df["tag"].fillna("").str.contains("main::rollover")
    is_release = df["kind"] == "release"
    boundaries = df[is_rollover & is_release]["t_us"].sort_values().values

    if len(boundaries) < 2:
        return pd.DataFrame()

    # 2. 提取所有 Worker 任务的完成时刻
    # 我们关注任务什么时候“完工”，通常以 complete 或 finished 为准
    excl = {"timer", "main::expand", "main::rollover"}
    worker_mask = ~df["tag"].fillna("").str.contains("|".join(excl))

    # 提取 Worker 任务的 release (开始算起) 和 complete (完工)
    worker_releases = df[worker_mask & (df["kind"] == "release")][["tid", "t_us", "tag"]]
    worker_completes = df[worker_mask & (df["kind"] == "complete")][["tid", "t_us"]]

    hp_results = []

    # 3. 遍历每一个超周期窗口
    for i in range(len(boundaries) - 1):
        b_start = boundaries[i]
        b_end = boundaries[i+1]

        # 找出在本周期内释放的任务
        jobs_released = worker_releases[(worker_releases["t_us"] >= b_start) &
                                        (worker_releases["t_us"] < b_end)]

        if not jobs_released.empty:
            # 找出这些任务对应的完成时间
            # 逻辑：对于每一个在本周期 release 的任务，寻找它之后最近的一个 complete
            # 简化逻辑：在本周期内或稍后一点点时间内的所有 worker complete 事件的最大值
            # 允许 10% 的超周期溢出，以捕捉跨周期的任务完成点
            buffer = (b_end - b_start) * 0.1
            relevant_completes = worker_completes[(worker_completes["t_us"] > b_start) &
                                                  (worker_completes["t_us"] < b_end + buffer)]

            if not relevant_completes.empty:
                last_complete = relevant_completes["t_us"].max()
                makespan = last_complete - b_start

                hp_results.append({
                    "hp_index": i,
                    "makespan": makespan,
                    "job_count": len(jobs_released)
                })

    return pd.DataFrame(hp_results)

In [124]:
import re

def aggregate_speedup_by_hp(root_dir):
    # 匹配文件名，例如 trace_fork_u10_m1_s10000.csv
    pattern = re.compile(
        r"trace_(?P<kind>\w+)_u(?P<u>\d+)_m(?P<m>\d+)_s(?P<seed>\d+).*\.csv"
    )
    results = []

    if not os.path.exists(root_dir):
        print(f"目录不存在: {root_dir}")
        return pd.DataFrame()

    files = [f for f in os.listdir(root_dir) if f.endswith(".csv")]
    print(f"正在扫描: {root_dir}，找到 {len(files)} 个 CSV 文件")

    for filename in sorted(files):
        match = pattern.match(filename)
        if match:
            path = os.path.join(root_dir, filename)
            df_hp = extract_hp_makespans(path)

            if not df_hp.empty:
                median_makespan = df_hp["makespan"].median()
                results.append({
                    "kind": match.group("kind"),
                    "u": match.group("u"),
                    "m": int(match.group("m")),
                    "T_hp": median_makespan
                })
            else:
                print(f"文件 {filename} 解析结果为空 (可能是缺少 rollover 或任务)")

    if not results:
        print("没有提取到任何有效数据，请检查正则匹配或 CSV 内容。")
        return pd.DataFrame()

    agg = pd.DataFrame(results)

    # 跨 seed 取中位数
    agg_median = agg.groupby(["kind", "u", "m"])["T_hp"].median().reset_index()

    # 获取 m=1 基准
    # 注意：这里需要根据 kind 和 u 匹配
    base = agg_median[agg_median["m"] == 1].copy()
    base = base.rename(columns={"T_hp": "T1"})[["kind", "u", "T1"]]

    if base.empty:
        print("警告：未找到 m=1 的基准数据，无法计算 Speedup。")
        return agg_median

    final_df = agg_median.merge(base, on=["kind", "u"], how="left")

    # 计算加速比 S = T1 / Tm
    final_df["Speedup"] = final_df["T1"] / final_df["T_hp"]
    final_df["Efficiency"] = (final_df["Speedup"] / final_df["m"]) * 100

    return final_df.sort_values(["kind", "u", "m"])

# 执行
agg_results = aggregate_speedup_by_hp("test")
print(agg_results)

正在扫描: test，找到 3 个 CSV 文件
   kind   u  m     T_hp       T1   Speedup  Efficiency
0  fork  10  1  83190.0  83190.0  1.000000  100.000000
1  fork  10  2  59761.0  83190.0  1.392045   69.602249
2  fork  10  3  30058.5  83190.0  2.767603   92.253439


## 1. 交互式加速比曲线 (Science Style)
说明：绘制 S(m) 随 m 增长的曲线，并叠加一条虚线作为 Ideal (Linear) Speedup 参考。

In [125]:
def draw_px_speedup_curve(agg_df):
    # 创建理想线性参考线数据
    m_range = agg_df["m"].unique()
    ideal_line = pd.DataFrame({"m": m_range, "Speedup": m_range, "Type": "Ideal (Linear)"})

    # 绘制实际数据
    fig = px.line(
        agg_df, x="m", y="Speedup", color="kind", symbol="kind",
        markers=True,
        title="System Speedup vs. Number of Workers",
        labels={"m": "Number of Worker Cores ($m$)", "Speedup": "Speedup $S(m)$"},
        color_discrete_sequence=[SCI_PALETTE["Active"], SCI_PALETTE["Post"]]
    )

    # 叠加理想线
    fig.add_scatter(x=ideal_line["m"], y=ideal_line["Speedup"],
                    mode='lines', name='Ideal Speedup',
                    line=dict(dash='dash', color=SCI_PALETTE["Idle"]))

    fig.update_layout(
        plot_bgcolor='white',
        xaxis=dict(gridcolor='#F0F0F0', dtick=1),
        yaxis=dict(gridcolor='#F0F0F0'),
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
        height=600
    )
    fig.show()

draw_px_speedup_curve(agg_results)

## 2. 并行效率分析图 (Scalability)
说明：展示 E(m) 随核心数增加的衰减情况。这是 Science 论文中用来讨论“收益递减”和“调度瓶颈”的标准图表。

In [106]:
def draw_px_efficiency_curve(agg_df):
    fig = px.line(
        agg_df, x="m", y="Efficiency", color="kind", markers=True,
        title="Parallel Efficiency Scalability",
        labels={"m": "Number of Workers ($m$)", "Efficiency": "Efficiency (%)"},
        color_discrete_sequence=[SCI_PALETTE["Overhead"], SCI_PALETTE["Wakeup"]]
    )

    # 添加 100% 效率参考线
    fig.add_hline(y=100, line_dash="dot", line_color="#000000", annotation_text="Ideal Scalability")

    fig.update_layout(
        plot_bgcolor='white',
        yaxis=dict(range=[0, 110], gridcolor='#F0F0F0'),
        xaxis=dict(gridcolor='#F0F0F0', dtick=1)
    )
    fig.show()

draw_px_efficiency_curve(agg_results)

## 3. 完工时间饱和分析 (Makespan Saturation)
说明：展示 Hp 绝对值的下降趋势。这能直观看到系统在核心数增加到多少时开始进入“平台期”。

In [108]:
def draw_px_makespan_saturation(agg_df):
    fig = px.bar(
        agg_df, x="m", y="T_hp", color="kind",
        barmode='group',
        title="Hyperperiod Makespan Reduction",
        labels={"T_hp": "Median Makespan (us)", "m": "Workers ($m$)"},
        color_discrete_sequence=[SCI_PALETTE["Active"], SCI_PALETTE["Post"]]
    )

    fig.update_layout(plot_bgcolor='white', yaxis=dict(gridcolor='#F0F0F0'))
    fig.show()

draw_px_makespan_saturation(agg_results)

### 加速比结果分析

1. **加速比饱和 (Saturation)**:
   - 如果曲线在 $m=4$ 后变平，说明系统受限于 **Amhdahl's Law**。即使核心再多，由于任务间的串行部分（如全局队列锁、主线程 `rollover`）无法并行，系统性能已达上限。

2. **效率衰减 (Efficiency Drop)**:
   - 如果 $E(m)$ 随 $m$ 增加剧烈下滑（例如从 90% 掉到 40%），说明 **调度开销 (Overhead)** 随核心数呈非线性增长。这通常是由于缓存一致性流量（Cache Coherence Traffic）或互斥锁争用导致的。

3. **释放受限下界 (Release-limited Bound)**:
   - 观察 `Makespan Reduction` 图。如果 $T_{hp}$ 最终接近最长算法节点的单次执行时间，说明你已经消除了所有调度空隙，达到了物理极限。